# Skin Lab — How does code change a photograph?

You will build an image-processing pipeline from five small functions. Start with a **7 × 7 pixel image**, where every
pixel is just three numbers. Then use **NumPy**, **SciPy**, and **Pillow** to run the same calculations across a full image.
At the end, **MediaPipe Face Mesh** adds a face boundary so the program changes pixels only where it is allowed.

### Your goal

By the end of the lab, you will be able to:

- explain how RGB numbers make a colour;
- turn a colour rule into a black-and-white mask;
- use a 3 × 3 kernel to count or blend neighbouring pixels;
- find a red spot by comparing it with its local area;
- combine masks to smooth texture, soften red areas, and adjust brightness on one photograph.

### How you will know the pipeline works

Every observation cell gives you **numbers, an image or overlay, and a short explanation**. The numbers show the calculation.
The image shows which pixels were selected. The explanation connects those two pieces of evidence.

This is a lesson about image-processing algorithms. It is **not a diagnostic tool and it does not rate anyone's skin**.
Lighting, cameras, backgrounds, and different skin tones can all make a hand-written colour rule fail.

Your code, checked steps, and current place are saved automatically in this browser. A captured or uploaded image is never
stored in `localStorage`. Use **Download notebook** if you want to move your work to another computer.

## Start here

Run the next two cells. The first loads the visual tools. The second loads NumPy, SciPy, Pillow, and the constants used by
the five functions. Do not edit these two cells yet.

You do not need a personal photo for most of the lab. The 7 × 7 image, a drawn face, and three public-domain photographs
are already included. The camera is used only once, in the final optional test.

For each coding task, read all four lines before editing:

- **Given:** values or tools already supplied;
- **INPUT:** data that enters the function;
- **PROCESS:** the exact code operation to complete;
- **OUTPUT:** the number, shape, type, or image that proves your code works.

In [ ]:
import magic_mirror
magic_mirror.skin_intro()

In [ ]:
import numpy as np
from PIL import Image
from scipy import ndimage


SKIN_VOTE_KERNEL = (
    (1, 1, 1),
    (1, 1, 1),
    (1, 1, 1),
)

SOFTEN_KERNEL = (
    (1, 2, 1),
    (2, 4, 2),
    (1, 2, 1),
)

MASK_OFF, MASK_ON = 0, 255
SKIN_NEIGHBOURS_NEEDED = 5
PIMPLE_RED_GAP = 24

## First observation — what must the program produce?

Run the next cell before writing code. It shows one input image and three outputs:

1. `skin_mask`: white (`255`) means “this pixel may be part of the skin region”; black (`0`) means “do not select it.”
2. `pimple_mask`: white (`255`) marks a locally red area for stronger smoothing; black (`0`) keeps the original colour.
3. final image: only selected pixels receive a calculated replacement colour.

A mask is a location map, not a colour photograph. The numbers `0` and `255` do not describe a person's skin and are not
a score. Before you continue, predict this: **why should the program show both the mask and the final image?**

In [ ]:
magic_mirror.show_skin_pipeline_overview()

## Mechanism 1 — one pixel contains three RGB numbers

Open the interactive panel and select the centre pixel. Its colour is `(225, 62, 66)`, so `R = 225`, `G = 62`, and
`B = 66`. Move one slider at a time. Watch the colour swatch and the three channel swatches.

To keep only the red channel, the program keeps `R = 225` and sets the other two values to zero:

```text
(225, 62, 66) → (225, 0, 0)
```

Green-only and blue-only use the same operation. Complete the panel's prediction with a different set of numbers before
the matching Python code is revealed.

In [ ]:
magic_mirror.show_mechanism("rgb_pixel")

In [ ]:
magic_mirror.show_skin_pixel_channels()

## From one pixel to a whole image with NumPy

Before using a photograph, build a **3 × 3 colour matrix**. Each position stores an RGB triplet. The code cell below gives
you nine colours, including pure red, green, blue, and the three colours used later in the drawn face.

The panel selected a pixel by row and column. NumPy uses the same address system:

- `pixels[3, 3]` reads all three RGB values at row 3, column 3;
- `pixels[:, :, 0]` reads the red value at every row and column;
- `pixels[:, :, 1]` reads every green value;
- `pixels[:, :, 2]` reads every blue value.

For the 3 × 3 example, `pixels.shape` is `(3, 3, 3)`: 3 rows, 3 columns, and 3 channels. Separating the channels produces
three ordinary number matrices. At row 2, column 0, they contain `R = 183`, `G = 127`, and `B = 103`; putting those three
values back in the same order rebuilds the colour `(183, 127, 103)`.

Run the next two cells. First read the complete matrix and select one position. Then inspect six panels: the RGB matrix,
the R/G/B number matrices shown as coloured intensities, the rebuilt RGB matrix, and the difference matrix. A maximum
difference of `0` proves that all nine colours were rebuilt without losing a channel value.

In [ ]:
import numpy as np

pixels = np.array([
    [[255, 0, 0], [0, 255, 0], [0, 0, 255]],
    [[255, 255, 0], [0, 255, 255], [255, 0, 255]],
    [[183, 127, 103], [225, 62, 66], [35, 80, 185]],
], dtype=np.int16)

print("pixels shape:", pixels.shape, "= rows, columns, RGB channels")
print("pixel at row 2, column 0:", pixels[2, 0])
print("R matrix:\n", pixels[:, :, 0])
print("G matrix:\n", pixels[:, :, 1])
print("B matrix:\n", pixels[:, :, 2])

In [ ]:
magic_mirror.show_numpy_channels()

## The library operations used in this project

The interactive panels let you calculate one small example. The project uses library functions to repeat the same
operation at every pixel:

| Image operation | Library call |
|---|---|
| Multiply and add values in a 3 × 3 area | `scipy.ndimage.convolve` |
| Find the mean of a 5 × 5 area | `scipy.ndimage.uniform_filter` |
| Expand one selected pixel into a 3 × 3 area | `scipy.ndimage.maximum_filter` |
| Choose the new or original RGB value at each pixel | `np.where` |
| Keep channel values between 0 and 255 | `np.clip` |
| Turn an array back into a picture | `Image.fromarray` |

You do not need a Python loop for every row and column. Your job is to send the correct arrays into each library function,
then check the result with both numbers and images.

## Investigation 1 — turn RGB evidence into 0 or 255

Start with the pixel `(183, 127, 103)`. Substitute those values into the three calculations:

```text
brightness = (183 + 127 + 103) // 3 = 413 // 3 = 137
warmth = 183 - 103 = 80
red_green_gap = 183 - 127 = 56
```

All three results satisfy the conditions in the starter code, so this pixel receives `255` and appears white in the mask.

Now test the blue background `(35, 80, 185)`:

```text
warmth = 35 - 185 = -150
-150 >= 8 → False → mask value 0
```

This is a deliberately simple RGB rule for learning the mechanism. It will not identify every skin tone under every kind
of lighting. Later, public photographs will help you find its limits.

In [ ]:
magic_mirror.show_mechanism("rgb_rule")

### Coding task 1 — complete `skin_evidence`

- **Given:** `red`, `green`, and `blue`, either as three numbers or three arrays with the same shape.
- **INPUT:** no outside input yet; later `detect_skin` will pass in the three channels of an image.
- **PROCESS:** calculate `warmth = red - blue` and `red_green_gap = red - green`, then pass `looks_like_skin` to `np.where`.
- **OUTPUT:** `(183, 127, 103)` must return `255`; `(35, 80, 185)` must return `0`. Array input must produce a
  two-dimensional `uint8` array with the same height and width.

In [ ]:
def skin_evidence(red, green, blue):
    """Apply one RGB rule to a pixel or to three complete NumPy channels."""
    red = np.asarray(red, dtype=np.int16)
    green = np.asarray(green, dtype=np.int16)
    blue = np.asarray(blue, dtype=np.int16)

    # TASK 2.
    # Calculate brightness, warmth, and red_green_gap.
    # Use &, not and, because each condition applies across an entire array.
    brightness = (red + green + blue) // 3
    warmth = ___
    red_green_gap = ___
    looks_like_skin = (
        (brightness >= 35) & (brightness <= 240)
        & (warmth >= 8)
        & (red_green_gap >= -10) & (red_green_gap <= 90)
    )
    result = np.where(___, MASK_ON, MASK_OFF).astype(np.uint8)
    return int(result) if result.ndim == 0 else result

In [ ]:
magic_mirror.preview_skin_evidence()

## Investigation 2 — use nearby pixels to repair one uncertain decision

The centre red pixel fails the RGB rule, so its raw mask value is `0`. The eight nearby pixels pass and have value `255`.
Before counting, the program changes `255` to `1` and leaves `0` as `0`:

```text
count = 1 + 1 + 1 + 1 + 0 + 1 + 1 + 1 + 1 = 8
8 >= 5 → True → centre skin_mask value = 255
```

The centre stays inside the selected region because 8 of the 9 nearby decisions pass the rule. This is a cause-and-effect
choice: lowering the required count selects more pixels; raising it selects fewer. Change the threshold in the panel and
observe the exact count before moving on.

In [ ]:
magic_mirror.show_mechanism("neighbours")

## How a 3 × 3 kernel calculates one new value

`convolve_layer` centres a 3 × 3 kernel on the pixel being calculated. It multiplies each image value by the kernel value
in the same position, adds the nine products, and then divides by `divisor`.

Suppose the eight outer values are `10`, the centre is `90`, and all nine kernel values are `1`:

```text
total = 8 × 10 + 1 × 90 = 170
new_value = 170 / 9 = 18.89
```

The centre output is `18.89`, not just an unexplained number: it is the local average. The value `90` became less dominant
because it was blended with eight values of `10`.

`ndimage.convolve` performs this calculation at every pixel. `mode="nearest"` handles an edge by reusing the value of the
nearest border pixel when part of the 3 × 3 area would fall outside the image.

In [ ]:
magic_mirror.show_convolution_math()

### Coding task 2 — complete `convolve_layer`

- **Given:** `layer`, `kernel`, and `divisor`; NumPy and SciPy are already imported.
- **INPUT:** the grader supplies a 5 × 5 array; there is no camera or file input in this task.
- **PROCESS:** call `ndimage.convolve(values, weights, mode="nearest")`, then divide the returned array by `divisor`.
- **OUTPUT:** if only the centre input is `9`, all nine weights are `1`, and `divisor = 9`, the centre output must be `1`.
  The original input must still contain its centre value `9`.

In [ ]:
def convolve_layer(layer, kernel, divisor):
    """Apply a SciPy kernel and return a new NumPy array."""
    # TASK 1.
    # Given: layer, kernel, and divisor.
    # 1. Convert layer and kernel to NumPy arrays with dtype np.float32.
    # 2. Call ndimage.convolve(values, weights, mode="nearest").
    # 3. Divide the returned array by divisor and return it.
    values = np.asarray(layer, dtype=np.float32)
    weights = np.asarray(kernel, dtype=np.float32)
    filtered = ndimage.convolve(___, ___, mode="nearest")
    return ___ / divisor

In [ ]:
magic_mirror.preview_library_convolution()

### Coding task 3 — complete `detect_skin`

- **Given:** a PIL image `img`, a 3 × 3 kernel of ones, and a required neighbour count of `5`.
- **INPUT:** one image; in the final task it can be a captured or uploaded photograph.
- **PROCESS:** change `raw_mask` from `0/255` into `binary` values `0/1`; call `convolve_layer` to count the 3 × 3 area;
  compare `neighbour_count` with `SKIN_NEIGHBOURS_NEEDED`.
- **OUTPUT:** a two-dimensional `uint8` array containing only `0` and `255`. The red centre of the drawn skin region must
  be `255`, while the centre of a solid blue image must be `0`.

In [ ]:
def detect_skin(img):
    """Create a skin-region mask by counting decisions in each 3x3 area."""
    # TASK 3.
    # 1. Convert the PIL image to a pixels array with shape (height, width, 3).
    # 2. Pass the three channels to skin_evidence to create raw_mask.
    # 3. Change 255 to 1, then count passing pixels in each 3x3 area.
    # 4. Use np.where to create a skin_mask containing only 0 and 255.
    pixels = np.asarray(img.convert("RGB"), dtype=np.int16)
    raw_mask = skin_evidence(
        pixels[:, :, 0],
        pixels[:, :, 1],
        pixels[:, :, 2],
    )
    binary = (raw_mask == MASK_ON).astype(np.float32)
    neighbour_count = convolve_layer(___, SKIN_VOTE_KERNEL, 1)
    return np.where(___ >= SKIN_NEIGHBOURS_NEEDED, MASK_ON, MASK_OFF).astype(np.uint8)

In [ ]:
magic_mirror.preview_skin_mask()

## Investigation 3 — find a pixel that is redder than its local area

A high red channel alone is not enough: an entire photograph might have warm or red lighting. Instead, compare each pixel
with the 5 × 5 area around it.

For the red pixel `(225, 62, 66)`:

```text
redness_spot = 225 - (62 + 66) / 2 = 225 - 64 = 161
```

For a nearby skin-coloured pixel `(183, 127, 103)`:

```text
redness_skin = 183 - (127 + 103) / 2 = 183 - 115 = 68
```

If the 5 × 5 area contains one red pixel and 24 surrounding pixels, then:

```text
local_redness = (161 + 24 × 68) / 25 = 1793 / 25 = 71.72
red_gap = 161 - 71.72 = 89.28
89.28 >= 24 → True → select the centre pixel
```

Finally, `maximum_filter` expands one selected pixel into a 3 × 3 area. That lets the later blend include nearby colour,
instead of changing only one isolated dot.

In [ ]:
magic_mirror.show_mechanism("red_spot")

### Coding task 4 — complete `detect_pimples`

- **Given:** RGB image `img`, its `skin_mask`, a 5 × 5 local area, and threshold `24`.
- **INPUT:** one RGB image and the matching two-dimensional skin mask.
- **PROCESS:** pass `redness` to `uniform_filter`; pass the Boolean `candidate` array to `maximum_filter`.
- **OUTPUT:** a `uint8` `pimple_mask`. The centre of the red test area must equal `255`; the image corner must equal `0`.

In [ ]:
def detect_pimples(img, skin_mask):
    """Find a locally red spot in a 5x5 area, then expand the selection."""
    # TASK 4.
    # uniform_filter calculates the mean of each 5x5 area.
    # maximum_filter expands a selected location to its nearby pixels.
    pixels = np.asarray(img.convert("RGB"), dtype=np.float32)
    red, green, blue = pixels[:, :, 0], pixels[:, :, 1], pixels[:, :, 2]
    redness = np.maximum(0, red - (green + blue) / 2)
    local_redness = ndimage.uniform_filter(___, size=5, mode="nearest")
    candidate = (
        (np.asarray(skin_mask) == MASK_ON)
        & (redness - local_redness >= PIMPLE_RED_GAP)
    )
    expanded = ndimage.maximum_filter(___, size=3, mode="nearest")
    return np.where(expanded, MASK_ON, MASK_OFF).astype(np.uint8)

In [ ]:
magic_mirror.preview_pimple_mask()

## Investigation 4 — calculate a replacement colour, then choose where to use it

The smoothing kernel gives the centre pixel weight `4`, its four side neighbours weight `2`, and its four diagonal
neighbours weight `1`. The nine weights add to `16`:

```text
1  2  1
2  4  2      weight total = 16
1  2  1
```

The centre is `(225, 62, 66)` and all eight neighbours are `(183, 127, 103)`. Calculate each channel separately:

```text
new_red   = (4 × 225 + 12 × 183) / 16 = 3096 / 16 = 193.5 → 194
new_green = (4 ×  62 + 12 × 127) / 16 = 1772 / 16 = 110.75 → 111
new_blue  = (4 ×  66 + 12 × 103) / 16 = 1500 / 16 = 93.75 → 94
```

The calculated colour is `(194, 111, 94)`. Then `np.where` makes a separate decision at each location:

- `pimple_mask == 255` → use `(194, 111, 94)`;
- `pimple_mask == 0` → keep the original `(225, 62, 66)`.

The program may calculate a smoothed version of the whole image, but the mask controls where that version is visible.

In [ ]:
magic_mirror.show_mechanism("soften")

### Coding task 5 — complete `remove_pimples`

- **Given:** image `img`, the smoothing kernel, and the four functions you completed above.
- **INPUT:** one PIL image; the final task can pass in a captured or uploaded photograph.
- **PROCESS:** pass `pixels` to `ndimage.convolve`; use `pimple_mask` as the condition in `np.where`.
  `pimple_mask[:, :, None]` repeats one location decision for the R, G, and B values at that pixel.
- **OUTPUT:** a PIL image with the same size. The red centre must become less different from its neighbours, the corner
  must stay unchanged, and the function must not edit the input image in place.

In [ ]:
def remove_pimples(img):
    """Smooth where pimple_mask is 255 and keep every other pixel unchanged."""
    # TASK 5.
    # 1. Create skin_mask and pimple_mask with the previous two functions.
    # 2. Add one dimension so the kernel shape is (3, 3, 1); this keeps R, G, B separate.
    # 3. ndimage.convolve smooths all three channels in one call.
    # 4. np.where uses the smooth colour only where pimple_mask equals 255.
    source = img.convert("RGB")
    pixels = np.asarray(source, dtype=np.float32)
    skin_mask = detect_skin(source)
    pimple_mask = detect_pimples(source, skin_mask)

    weights = np.asarray(SOFTEN_KERNEL, dtype=np.float32)[:, :, None]
    softened = ndimage.convolve(___, weights, mode="nearest") / weights.sum()
    combined = np.where(___[:, :, None] == MASK_ON, softened, pixels)
    output = np.clip(np.rint(combined), 0, 255).astype(np.uint8)
    return Image.fromarray(output, "RGB")

In [ ]:
magic_mirror.preview_cleanup()

## Check all five functions

Run the grader. Each line names a function and explains any failing result. The final line must read:

```text
Result: 5/5 parts correct.
```

The page saves your code, grader progress, and the six mechanism panels, so you can continue later on this computer.

In [ ]:
magic_mirror.check_skin_code()

## Connect the five functions into one visible pipeline

Run the next cell. The six labelled images follow the actual data path:

```text
RGB input → skin_mask → skin overlay → pimple_mask → red-area overlay → output image
```

The overlays reveal the exact selected locations. If a location is wrong, inspect the RGB calculations, the 3 × 3 count,
or the local red difference. If the location is correct but the output colour is wrong, inspect the kernel calculation and
the condition passed to `np.where`.

In [ ]:
magic_mirror.skin_demo()

## Explore three more NumPy filters and three kernels

Run the next two cells and compare every labelled image with its input. Record answers to these questions:

1. Which operations use only the RGB values at the current pixel?
2. Which operations require values from neighbouring pixels?
3. In the sharpening kernel, what number multiplies the centre pixel?
4. Why can an edge image contain bright lines even when the input has no white line there?

In [ ]:
magic_mirror.numpy_filter_gallery()

In [ ]:
magic_mirror.numpy_kernel_gallery()

### Modify a NumPy colour filter

The starter function adds `40` to the blue channel and uses `np.clip` to keep every value in `0..255`.

- **Given:** a sample RGB NumPy array.
- **INPUT:** no outside input in this task.
- **PROCESS:** copy the array, change exactly one channel, clip the values, and return a `uint8` result.
- **OUTPUT:** a before/after figure plus the result's shape and data type. The input array must remain unchanged.

Run it once. Then change the channel index or the added amount and state which visible change your new numbers caused.

In [ ]:
def my_numpy_filter(pixels):
    result = pixels.copy().astype(np.int16)
    result[:, :, 2] = np.clip(result[:, :, 2] + 40, 0, 255)
    return result.astype(np.uint8)

magic_mirror.preview_numpy_filter(my_numpy_filter)

## Test the simple rule on public photographs

Three CC0 images are bundled with the lesson, so the page does not hotlink personal data: portraits by
[William Stitt](https://commons.wikimedia.org/wiki/File:Face_portrait_(Unsplash).jpg) and
[Eddie Kopp](https://commons.wikimedia.org/wiki/File:Young_woman%27s_face_(Unsplash).jpg), plus a skin close-up by
[Montavius Howard](https://commons.wikimedia.org/wiki/File:Human_skin_close-up.jpg).

Run `try_public_photo(0)`, then change the index to `1` and `2`. Each run shows the input, the skin overlay, the red-area
overlay, and the output. Use the printed pixel counts and the overlays as evidence:

- Where did the colour rule miss part of the intended region?
- Where did it select a background or feature by mistake?
- Which change could be caused by lighting rather than the subject?

The goal is to test the limits of your code, not to make a claim about any person in the photographs.

In [ ]:
magic_mirror.show_public_photo_gallery()

In [ ]:
magic_mirror.try_public_photo(0)

## Mechanism 6 — require both masks before changing a pixel

The RGB rule may select an object whose colour is similar to a skin tone. Face Mesh adds a second question: is this pixel
inside the face boundary?

- `face_mask = 1`: the pixel lies inside the face outline;
- `skin_mask = 1`: the pixel passed the colour and neighbour checks.

The program calculates `allowed = face_mask & skin_mask`. Only `1 & 1` produces `1`; the other three combinations keep
the original colour. Use the panel to test all four combinations, then complete its new prediction.

In [ ]:
magic_mirror.show_mechanism("face_gate")

## How MediaPipe creates `face_mask`

MediaPipe Face Mesh receives one image and returns up to 478 landmark points on a detected face. Each point contains a
horizontal and vertical position. The browser selects points around the outer face, including point `10` near the forehead,
`454` on the right, `152` near the chin, and `234` on the left. It joins the boundary points into a closed shape and fills
the inside with `1`; the outside remains `0`.

```text
face_mask = pixel lies inside the face outline
skin_mask = pixel passes the colour and neighbour checks
allowed   = face_mask & skin_mask
output    = np.where(allowed[..., None], cleaned, original)
```

`allowed[..., None]` applies the same allowed/not-allowed decision to all three RGB values. Face Mesh supplies a location
boundary. It does not diagnose skin and it does not find red spots by itself.

In [ ]:
magic_mirror.show_face_mesh_map()

In [ ]:
magic_mirror.show_face_mask_pipeline()

## Capstone — build a visible, adjustable portrait pipeline

Your five functions are the foundation. A real photograph needs three additional controls so the result is visible without
smearing strong facial details:

1. Face Mesh creates `face_mask`. Your RGB mask is combined with Y/Cb/Cr colour evidence. `Y` tracks lightness while `Cb`
   and `Cr` track blue and red colour differences, so the decision is not tied to one exact RGB triplet.
2. `scipy.ndimage.sobel` measures rapid changes in lightness. Strong edges often belong to eyes, lips, hair, or the face
   outline, so the pipeline protects those locations from general smoothing.
3. Your chosen 3 × 3 kernel smooths the remaining region. Red areas receive a stronger blend. A small brightness value is
   added only inside the allowed mask.

For one channel, suppose the original value is `200`, the kernel result is `188`, and `skin_smooth_strength = 0.55`:

```text
mixed = 200 × (1 - 0.55) + 188 × 0.55
      = 200 × 0.45 + 188 × 0.55
      = 193.4 → 193
bright = 193 + 10 = 203
```

The kernel decides the candidate smooth colour. `skin_smooth_strength` decides how much of that colour to use.
`skin_brightness` adds a controlled offset after blending.

### Capstone task — choose and defend your settings

- **Given:** three 3 × 3 kernels and safe ranges for all settings.
- **INPUT:** the next cell uses a public portrait; the final cell accepts one captured or uploaded photograph.
- **PROCESS:** change `kernel_choice`, then test one or more strengths, brightness, pass count, or the nine kernel weights.
- **OUTPUT:** the report must name the kernel, weight total, strengths, and changed-pixel count. The five-panel figure must
  show input, skin region, red region, magnified difference, and final output.

Change one setting at a time. Use the difference panel and changed-pixel count—not just “it looks better”—to explain the
effect of your change.

In [ ]:
# Choose "gentle", "balanced", or "strong".
kernel_choice = "balanced"

# You may also edit any of the nine weights.
kernel_options = {
    "gentle": (
        (1, 2, 1),
        (2, 4, 2),
        (1, 2, 1),
    ),
    "balanced": (
        (1, 1, 1),
        (1, 1, 1),
        (1, 1, 1),
    ),
    "strong": (
        (1, 1, 1),
        (1, 0, 1),
        (1, 1, 1),
    ),
}

SOFTEN_KERNEL = kernel_options[kernel_choice]
skin_smooth_strength = 0.55   # 0.00 keeps the input; 1.00 uses the full smooth colour
spot_smooth_strength = 0.90   # red areas receive a stronger blend
skin_brightness = 10          # add -25 to 25 inside the selected skin region
skin_kernel_passes = 2        # run the kernel 1 to 4 times
redness_sensitivity = 1.6     # lower selects more red areas; higher selects fewer

magic_mirror.describe_skin_pipeline_settings()

In [ ]:
magic_mirror.preview_pro_skin_pipeline()

## Run the pipeline once on a photograph

- **Given:** your five functions, your current capstone settings, and MediaPipe's face-boundary landmark list.
- **INPUT:** exactly one captured photograph. If the camera is unavailable, select a JPG, PNG, or WebP file from this device.
- **PROCESS:** run the cell, frame one face, and press **Capture one photo**. The camera stops immediately. Face Mesh runs
  once; NumPy and SciPy then run the pipeline once on that still image.
- **OUTPUT:** five panels show the input, allowed skin region, stronger red region, magnified colour difference, and final
  result. The report states how many pixels were selected, protected, and changed, plus your kernel and settings.

The image stays only in this cell's visible output. It is not written to `localStorage`, and it disappears after a reload.
Your code and progress remain. Processing uses 320 × 240 pixels and displays at 480 × 360 for a clear still-image result.

In [ ]:
magic_mirror.capture_skin_photo()

## Final explanation — claim, evidence, reasoning

Add a code or text cell and write four short statements:

1. **Claim:** name one setting that changed the result in a useful, visible way.
2. **Evidence:** give the before/after setting values and the reported changed-pixel count.
3. **Reasoning:** explain how the kernel, mask, or blend caused that change.
4. **Limitation:** identify one missed or wrongly selected area and explain why lighting, colour, or the face boundary may
   have caused it.

Your explanation is complete only when it cites a number and a visible panel from your own run.